# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. The dataset is defined by a Croissant schema and includes clinical and molecular records for cancer survivors with second primary colorectal cancer.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_object = dataset.metadata

# Display dataset metadata
print(f"Dataset Title: {metadata_object.name}")
print(f"Description: {metadata_object.description}")
print(f"Published: {getattr(metadata_object, 'datePublished', 'N/A')}")
print(f"Sample Size: 77 records (see below)")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get record sets from the Croissant metadata
record_sets = getattr(metadata_object, 'recordSet', [])

if not record_sets:
    print("No record sets are explicitly listed in metadata. Let's check what record sets are discoverable via mlcroissant.")
    # mlcroissant exposes available record_sets via dataset.available_record_sets
    avail_record_sets = dataset.available_record_sets()
    print("Discovered Record Set @ids:")
    for rset in avail_record_sets:
        print(f"- {rset}")
else:
    print("Record Sets declared in metadata:")
    for rset in record_sets:
        print(f"- {rset}")

# For demonstration, let's enumerate a sample from each record set
avail_record_sets = dataset.available_record_sets()
for rset in avail_record_sets:
    print(f"\nSample records from RecordSet @id: {rset}")
    try:
        records = list(dataset.records(record_set=rset))
        print("Fields in this RecordSet:")
        if records:
            print(list(records[0].keys()))
            print("Sample record:")
            print(records[0])
        else:
            print("No records found.")
    except Exception as e:
        print(f"Could not read records: {e}")

## 3. Data Extraction
Load data from all discovered record sets into DataFrames for analysis. Record sets and fields are referenced by their `@id`s.

In [ ]:
# Extract data from each record set
record_sets_to_load = dataset.available_record_sets()
dataframes = {}

for rs_id in record_sets_to_load:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nDataFrame columns for record set @id: {rs_id}")
        print(df.columns.tolist())
        print(df.head(2))
    else:
        print(f"\nNo records loaded for record set @id: {rs_id}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalizing, and grouping using `@id` references for fields.

In [ ]:
# Choose the largest record set loaded, and select numeric fields

# Identify which record set has most records
largest_rs_id = None
max_records = 0
for rs_id, df in dataframes.items():
    if len(df) > max_records:
        largest_rs_id = rs_id
        max_records = len(df)

if largest_rs_id is None:
    print("No record sets are loaded for analysis.")
else:
    df = dataframes[largest_rs_id]
    print(f"Using Record Set @id: {largest_rs_id} with {len(df)} records for EDA.")

    # Print available columns (which serve as field @ids)
    print("Available fields (@ids) in this Record Set:")
    print(df.columns.tolist())

    # Heuristically choose numeric columns (e.g., Age, DiagnosisInterval, assuming typical clinical fields)
    numeric_candidates = [col for col in df.columns if (
        df[col].dtype == 'int64' or df[col].dtype == 'float64') and not ('id' in col.lower())]
    if not numeric_candidates:
        numeric_candidates = [col for col in df.columns if 'Age' in col or 'Interval' in col]
    print("Numeric field candidates:", numeric_candidates)
    
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]

        # Filter records (e.g., Age > 50 or Interval > threshold)
        threshold = 50 if 'Age' in numeric_field_id else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize this numeric field
        norm_colname = f"{numeric_field_id}_normalized"
        filtered_df[norm_colname] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_colname]].head())

        # Group records by categorical attribute e.g. Sex, MSI_Status, TumorLocation
        group_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped average {numeric_field_id} by {group_field_id}:\n{grouped_df}")
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for filtering and normalization in this RecordSet.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Basic visualization of numeric field distributions
import matplotlib.pyplot as plt

if largest_rs_id and numeric_candidates:
    fig, ax = plt.subplots(figsize=(7,4))
    df[numeric_field_id].hist(ax=ax, bins=10, color='skyblue')
    ax.set_title(f"Distribution of {numeric_field_id} in RecordSet {largest_rs_id}")
    ax.set_xlabel(numeric_field_id)
    ax.set_ylabel("Frequency")
    plt.show()

    # If possible, visualize average numeric field per group
    if group_candidates:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean()
        grouped.plot(kind='bar', color='salmon')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded metadata and records from FAIR^2 CRC survivor dataset with clinical and molecular attributes.
- Identified available record sets and key fields by their `@id`.
- Performed filtering and normalization of numeric field (e.g., Age at diagnosis or diagnosis interval) and grouped by key attributes such as anatomical location or molecular status.
- Visualized distributions and group averages for exploratory purposes.

This FAIR-compliant dataset supports robust clinical and pathology-level analysis for cancer research and can be further processed for machine learning tasks or statistical studies.